# Contract Intelligence Platform — Config Setup
Writes `config.py` to your Google Drive project folder.
Run this notebook **once** before running the data pipeline notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/contract-intelligence-platform')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
(PROJECT_ROOT / 'src').mkdir(exist_ok=True)

config_content = '''
"""
config.py — Central configuration for Contract Intelligence Platform.
Colab-compatible: ROOT_DIR is set to Google Drive mount point when running in Colab.
"""

from pathlib import Path

try:
    import google.colab  # noqa: F401
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    ROOT_DIR = Path("/content/drive/MyDrive/contract-intelligence-platform")
else:
    ROOT_DIR = Path(__file__).parent.resolve()

DATA_DIR      = ROOT_DIR / "data"
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
CUAD_DIR      = DATA_DIR / "cuad"
LEDGAR_DIR    = DATA_DIR / "ledgar"
MAUD_DIR      = DATA_DIR / "maud"
EDGAR_DIR     = DATA_DIR / "edgar"
EDGAR_RAW_DIR = EDGAR_DIR / "raw"

MODELS_DIR                   = ROOT_DIR / "models"
CLASSIFIER_DIR               = MODELS_DIR / "clause_classifier"
CLASSIFIER_LEGALBERT_DIR     = CLASSIFIER_DIR / "legalbert"
CLASSIFIER_DEBERTA_DIR       = CLASSIFIER_DIR / "deberta"
CLASSIFIER_PRODUCTION_CONFIG = CLASSIFIER_DIR / "production_config.json"
ANOMALY_DIR                  = MODELS_DIR / "anomaly_detector"
POWER_DIR                    = MODELS_DIR / "power_scorer"

LOGS_DIR      = ROOT_DIR / "logs"
STATIC_DIR    = ROOT_DIR / "static"
TEMPLATES_DIR = ROOT_DIR / "templates"

DB_PATH = PROCESSED_DIR / "contracts.db"
DB_URL  = f"sqlite:///{DB_PATH}"

LEGAL_BERT_MODEL  = "nlpaueb/legal-bert-base-uncased"
DEBERTA_MODEL     = "microsoft/deberta-v3-base"
SENTIMENT_MODEL   = "cardiffnlp/twitter-roberta-base-sentiment-latest"
CUAD_DATASET_ID   = "theatticusproject/cuad-qa"
LEDGAR_DATASET_ID = "coastalcph/lex_glue"
MAUD_DATASET_ID   = "theatticusproject/maud"

# EDGAR settings
EDGAR_AUTO_LABEL_CONFIDENCE = 0.70
EDGAR_DOWNLOAD_LIMIT        = 5000
EDGAR_CLUSTER_N             = 20
EDGAR_REVIEW_PATH           = PROCESSED_DIR / "review_queue.csv"
EDGAR_NEW_TYPES_PATH        = PROCESSED_DIR / "edgar_new_clause_types.csv"

# BERTopic cluster discovery
BERTOPIC_MIN_CLUSTER_SIZE  = 10
BERTOPIC_UMAP_COMPONENTS   = 5
BERTOPIC_TOP_N_WORDS       = 10

# Cluster routing thresholds (cosine similarity to existing type name embeddings)
CLUSTER_ROUTE_HIGH_SIM     = 0.80
CLUSTER_ROUTE_LOW_SIM      = 0.50

# Taxonomy auto-expansion
TAXONOMY_AUTO_ADD_MIN_SIZE = 50
TAXONOMY_REVIEW_MIN_SIZE   = 10
TAXONOMY_REVIEW_PATH       = PROCESSED_DIR / "taxonomy_review.csv"
DYNAMIC_TAXONOMY_PATH      = PROCESSED_DIR / "dynamic_taxonomy.json"

# Unified clause taxonomy — 100 types
CUAD_CLAUSE_TYPES = [
    # CUAD core (41)
    "Competitive Restriction Exception", "Non-Compete", "Exclusivity",
    "No-Solicit Of Customers", "No-Solicit Of Employees", "Non-Disparagement",
    "Termination For Convenience", "Rofr/Rofo/Rofn", "Change Of Control",
    "Anti-Assignment", "Revenue/Profit Sharing", "Price Restrictions",
    "Minimum Commitment", "Volume Restriction", "IP Ownership Assignment",
    "Joint IP Ownership", "License Grant", "Non-Transferable License",
    "Affiliate License-Licensor", "Affiliate License-Licensee",
    "Unlimited/All-You-Can-Eat-License", "Irrevocable Or Perpetual License",
    "Source Code Escrow", "Post-Termination Services", "Audit Rights",
    "Uncapped Liability", "Cap On Liability", "Liquidated Damages",
    "Warranty Duration", "Insurance", "Covenant Not To Sue",
    "Third Party Beneficiary", "Most Favored Nation", "Governing Law",
    "Dispute Resolution", "Limitations Of Liability", "Indemnification",
    "Agreement Date", "Effective Date", "Expiration Date", "Renewal Term",
    # New from LEDGAR (32)
    "Confidentiality", "Amendments", "Anti-Corruption Laws",
    "Approvals And Consents", "Authority", "Compliance With Laws",
    "Consent To Jurisdiction", "Definitions", "Enforceability",
    "Entire Agreements", "Expenses", "Force Majeure", "Further Assurances",
    "Interests", "Liens And Encumbrances", "Limitations Of Remedies",
    "Non-Waiver", "Notices", "Organization And Existence", "Payments",
    "Representations And Warranties", "Sanctions", "Securities Law Compliance",
    "Severability", "Specific Performance", "Successors And Assigns",
    "Survival", "Taxes And Withholding", "Trade Controls",
    "Transactions With Affiliates", "Waiver Of Jury Trial", "Waivers",
    # New from MAUD (14)
    "No-Shop", "Material Adverse Effect", "Hell-Or-High-Water",
    "Termination Fee", "Reverse Termination Fee", "Matching Rights",
    "Superior Offer Definition", "Fiduciary Exception",
    "Antitrust Efforts Standard", "Operating Covenants",
    "Expense Reimbursement", "Intervening Event Definition",
    "Board Recommendation Change", "Tail Period",
    # Additional commercial provisions (13)
    "Data Protection And Privacy", "Employment And Benefits", "Capitalization",
    "Publicity And Announcements", "Non-Solicitation General", "Step-In Rights",
    "Benchmarking Rights", "Electronic Signatures And Counterparts",
    "Service Level Agreement", "Subcontracting", "Assignment Of Receivables",
    "Performance Bonds", "Escrow And Holdback",
]

# Dynamic taxonomy — auto-discovered types are persisted in dynamic_taxonomy.json
# and merged at import time without ever editing config.py manually
if DYNAMIC_TAXONOMY_PATH.exists():
    import json as _json
    _dynamic_types = _json.loads(DYNAMIC_TAXONOMY_PATH.read_text())
    CUAD_CLAUSE_TYPES = CUAD_CLAUSE_TYPES + [
        t for t in _dynamic_types if t not in CUAD_CLAUSE_TYPES
    ]

NUM_CLAUSE_TYPES = len(CUAD_CLAUSE_TYPES)

CLASSIFIER_LEARNING_RATE  = 2e-5
CLASSIFIER_BATCH_SIZE     = 16
CLASSIFIER_EPOCHS         = 25
CLASSIFIER_WARMUP_STEPS   = 500
CLASSIFIER_MAX_LENGTH     = 512
CLASSIFIER_THRESHOLD      = 0.5
CLASSIFIER_TARGET_F1      = 0.80
UNKNOWN_PROB_THRESHOLD    = 0.3

ISOLATION_FOREST_CONTAMINATION = 0.05
AUTOENCODER_HIDDEN_DIMS        = [256, 64]
AUTOENCODER_EPOCHS             = 30
AUTOENCODER_BATCH_SIZE         = 64
AUTOENCODER_LR                 = 1e-3
EMBEDDING_DIM                  = 768
ANOMALY_IF_WEIGHT              = 0.5
ANOMALY_AE_WEIGHT              = 0.5
ANOMALY_FLAG_THRESHOLD         = 70

POWER_WEIGHT_SENTIMENT     = 0.30
POWER_WEIGHT_MODAL_VERBS   = 0.25
POWER_WEIGHT_OBLIGATIONS   = 0.25
POWER_WEIGHT_ASSERTIVENESS = 0.20
OBLIGATION_MODALS  = {"shall", "must", "will", "required", "obligated"}
DISCRETION_MODALS  = {"may", "might", "could", "should", "permitted", "entitled"}
IMBALANCE_HIGH_THRESHOLD   = 40
IMBALANCE_MEDIUM_THRESHOLD = 20

SHAP_BACKGROUND_SAMPLES = 100
SHAP_MAX_EVALS          = 500
SHAP_OUTPUT_DIR         = PROCESSED_DIR / "shap_plots"

MAX_PAGES         = 100
CLAUSE_MIN_TOKENS = 10
CLAUSE_MAX_TOKENS = 512

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
RANDOM_SEED = 42

API_HOST     = "0.0.0.0"
API_PORT     = 8000
API_TITLE    = "Contract Intelligence & Power Imbalance Platform"
API_VERSION  = "1.0.0"
CORS_ORIGINS = ["*"]

EVAL_RESULTS_PATH = ROOT_DIR / "evaluation_results.json"
MODEL_CARD_PATH   = ROOT_DIR / "model_card.md"

for _dir in [
    RAW_DIR, PROCESSED_DIR, CUAD_DIR, LEDGAR_DIR, MAUD_DIR,
    EDGAR_DIR, EDGAR_RAW_DIR,
    CLASSIFIER_DIR, CLASSIFIER_LEGALBERT_DIR, CLASSIFIER_DEBERTA_DIR,
    ANOMALY_DIR, POWER_DIR, LOGS_DIR, SHAP_OUTPUT_DIR,
]:
    _dir.mkdir(parents=True, exist_ok=True)
'''.strip()

(PROJECT_ROOT / 'config.py').write_text(config_content)
print(f'config.py written to {PROJECT_ROOT / "config.py"}')

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
import config
print('ROOT_DIR :', config.ROOT_DIR)
print('DB_PATH  :', config.DB_PATH)
print('Clause types:', config.NUM_CLAUSE_TYPES)
print('All directories created successfully.')